In [1]:
from pathlib import Path
import hashlib
import shutil   # built-in library for copying , moving and deleting files/folders

In [2]:
# Paths 
raw_dir = Path("../data/raw")
processed_dir = Path("../data/processed")

train_raw = raw_dir / "Training"
test_raw = raw_dir / "Testing"

classes = [
    "glioma_tumor",
    "meningioma_tumor",
    "no_tumor",
    "pituitary_tumor"
]

In [3]:
# processed directory structure
for dataset in ['Training', 'Validation', 'Testing']:
    
    for cls in classes:
        folder = processed_dir / dataset / cls 
        folder.mkdir(parents= True, exist_ok= True)

In [4]:
# creating image hash function - md5 hash 

def get_image_hash(image_path):
    
    hasher = hashlib.md5()        # creates md5 hash object
    
    with open(image_path, "rb") as f:
        hasher.update(f.read())              # hasher.update() -> gives the read binary data to md5 to calculate its hash
        
    return hasher.hexdigest()             # converts hash into readable hexadecimal string

In [5]:
#creating a temporary directory to store duplicate free training dataset

clean_train_dir = processed_dir / "clean_train"

for cls in classes:
    (clean_train_dir / cls).mkdir(parents= True, exist_ok= True)
    

In [6]:
# remove duplicates from training images
train_hashes = set()

duplicate_train_images = []

for cls in classes:
    source_folder = train_raw / cls 
    destination_folder = clean_train_dir / cls
    
    for image_path in source_folder.iterdir():   # .iterdir() goes through every item inside source_folder
        
        # if item is not a file , skip it
        if not image_path.is_file():
            continue
        
        image_hash = get_image_hash(image_path)
        
        if image_hash in train_hashes:
            
            duplicate_train_images.append(image_path)
            continue
        
        train_hashes.add(image_hash)
        
        # copying the unique images in destination folder
        shutil.copy2(                     # .copy2() copies the image and also keeps some file information from the original (metadata such as modification time)
            image_path,
            destination_folder / image_path.name
        )

In [7]:
print("Unique training images : ", len(train_hashes))
print("Duplicate training images that were removed : ",len(duplicate_train_images))

Unique training images :  2777
Duplicate training images that were removed :  93


In [8]:
# splitting clean training data into training and validation dataset

import random

random.seed(42)
validation_ratio = 0.20

for class_name in classes:
    source_folder = clean_train_dir / class_name
    
    images = [
        image for image in source_folder.iterdir()
        if image.is_file()
    ]
    
    random.shuffle(images)
    
    validation_count = int(len(images) * validation_ratio)
    
    val_images = images[:validation_count]
    train_images = images[validation_count:]
    
    # copy training images to processed/Training
    for image_path in train_images:
        
        destination = (processed_dir / "Training" / class_name / image_path.name)
        
        shutil.copy2(image_path, destination)
    
    #copy validation images to processed/validation
    for image_path in val_images:
        destination = (processed_dir / "Validation" / class_name / image_path.name)
        shutil.copy2(image_path, destination)

In [9]:
# checking duplicates in clean_train

clean_image_hashes = set()
duplicates = []

for class_name in classes:
    clean_train_dir = processed_dir / "clean_train" / class_name
    
    for image_path in clean_train_dir.iterdir():
        image_hash = get_image_hash(image_path)
        
        if image_hash in clean_image_hashes:
            duplicates.append(image_path)
            continue
        
        clean_image_hashes.add(image_hash)
        

print(len(duplicates))
            
        

0


In [10]:
# cleaning the testing dataset

clean_test_dir = processed_dir / "clean_test"

for class_name in classes:
    (clean_test_dir / class_name).mkdir(parents= True, exist_ok= True)

In [11]:
# finding duplicates inside test dataset

test_hashes = set()
duplicate_test_images = []

for class_name in classes:
    
    source_folder = test_raw / class_name
    destination_folder = clean_test_dir / class_name
    
    for image_path in source_folder.iterdir():
        if not image_path.is_file():
            continue
        
        image_hash = get_image_hash(image_path)
        
        if image_hash in test_hashes:
            
            duplicate_test_images.append(image_path)
            continue
        
        test_hashes.add(image_hash)
        
        shutil.copy2(
            image_path,
            destination_folder / image_path.name
        )

In [12]:
# checking

print("Unique testing images : ",len(test_hashes))
print("Duplicate testing images removed : ",len(duplicate_test_images))

Unique testing images :  314
Duplicate testing images removed :  80


In [13]:
# copying clean test data to processed / Testing

for class_name in classes:
    
    source_folder = clean_test_dir / class_name
    destination_folder = processed_dir / "Testing" / class_name
    
    for image_path in source_folder.iterdir():
        
        if image_path.is_file():
            
            shutil.copy2(
                image_path,
                destination_folder / image_path.name 
            )

In [14]:
# checking duplicates in training and validation datasets

processed_train_hashes = set()

for class_name in classes:
    
    folder = processed_dir / "Training" / class_name
    
    for image_path in folder.iterdir():
        
        if image_path.is_file():
            processed_train_hashes.add(
                get_image_hash(image_path)
            )
            
train_val_leaks = []

for class_name in classes:
    
    folder = processed_dir / "validation" / class_name
    
    for image_path in folder.iterdir():
        
        if not image_path.is_file():
            continue
        
        image_hash = get_image_hash(image_path)
        
        if image_hash in processed_train_hashes:
            train_val_leaks.append(image_path)
            

print("Train-validation leakage:", len(train_val_leaks))

Train-validation leakage: 0


In [15]:
# checking train test leakage

train_hashes = set()

for dataset in ['Training', 'Validation']:
    for class_name in classes:
        folder = processed_dir / dataset / class_name
        
        for image_path in folder.iterdir():
            
            if image_path in folder.iterdir():
                if image_path.is_file():
                    train_hashes.add(
                        get_image_hash(image_path)
                    )

In [16]:
# searching the test set

train_test_leaks = []

for class_name in classes:
    folder = processed_dir / 'Testing' / class_name
    
    for image_path in folder.iterdir():
        if not image_path.is_file():
            continue
        
        image_hash = get_image_hash(image_path)
        
        if image_hash in train_hashes:
            train_test_leaks.append(image_path)

In [17]:
# checking number of duplicate images in train and test set

print('Train/validation - test leakage : ', len(train_test_leaks))

Train/validation - test leakage :  64


In [18]:
# removing leaked test images

for image_path in train_test_leaks:
    image_path.unlink()
    
print('Leaked test images removed : ',len(train_test_leaks))

Leaked test images removed :  64


In [19]:
# verifying leakage again

remaining_leaks = []
for class_name in classes:
    folder = processed_dir / 'Testing' / class_name
    
    for image_path in folder.iterdir():
        if not image_path.is_file():
            continue
        
        image_hash = get_image_hash(image_path)
        
        if image_hash in train_hashes:
            remaining_leaks.append(image_path)
            
print("Remaining leakage:", len(remaining_leaks))

Remaining leakage: 0


In [21]:
# final dataset information

for dataset in ['Training', 'Validation', 'Testing']:
    print(f'\n{dataset.upper()}')
    print('-' * 30)
    
    total = 0
    
    for class_name in classes:
        folder = processed_dir / dataset / class_name
        
        count = sum(
            1 for image in folder.iterdir()
            if image.is_file()
        )
        
        total += count 
        
        print(f"{class_name} : {count}")
        
    print(f"Total : {total}")


TRAINING
------------------------------
glioma_tumor : 655
meningioma_tumor : 652
no_tumor : 268
pituitary_tumor : 648
Total : 2223

VALIDATION
------------------------------
glioma_tumor : 163
meningioma_tumor : 162
no_tumor : 67
pituitary_tumor : 162
Total : 554

TESTING
------------------------------
glioma_tumor : 84
meningioma_tumor : 113
no_tumor : 17
pituitary_tumor : 36
Total : 250
